#Setup and Config

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    current_timestamp,
    regexp_extract,
    col,
    lit
)
from pyspark.sql.types import StructType, StructField, StringType

spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()

# Paths
# On Databricks Community Edition use DBFS paths
CRICSHEET_SOURCE_PATH = "/Volumes/ipl_catalog/raw_data/cricsheet_files"
BRONZE_TABLE_PATH = "/Volumes/ipl_catalog/raw_data/bronze_deliveries/"
CHECKPOINT_PATH = "/Volumes/ipl_catalog/raw_data/checkpoints/bronze_deliveries/"
SCHEMA_PATH = "/Volumes/ipl_catalog/raw_data/checkpoints/schema/"
BRONZE_DB = "ipl_catalog"
BRONZE_TABLE = "ipl_catalog.bronze.raw_deliveries"

# Create catalogs and schemas and upload CSVs

In [0]:
# create catalog
spark.sql("CREATE CATALOG IF NOT EXISTS ipl_catalog")

# create raw_data, BRONZE, SILVER and GOLD schema(database)
spark.sql("CREATE SCHEMA IF NOT EXISTS ipl_catalog.raw_data")
spark.sql("CREATE SCHEMA IF NOT EXISTS ipl_catalog.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS ipl_catalog.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS ipl_catalog.gold")

# create a volume - this is like a folder in Unity Catalog
spark.sql("CREATE VOLUME IF NOT EXISTS ipl_catalog.raw_data.cricsheet_files")
spark.sql("CREATE VOLUME IF NOT EXISTS ipl_catalog.raw_data.checkpoints")
spark.sql("CREATE VOLUME IF NOT EXISTS ipl_catalog.raw_data.bronze")

# create directories for checkpoint
dbutils.fs.mkdirs(CHECKPOINT_PATH)
dbutils.fs.mkdirs(SCHEMA_PATH)

# Delivery Schema Definition

In [0]:
"""
Always define schema explicitely with AutoLoader -never let it infer on streaming read
"""

# Cricsheet CSV2 delivery format columns
delivery_schema = StructType([
    StructField('match_id',     StringType(), True),
    StructField('season',       StringType(), True),
    StructField('start_date',   StringType(), True),
    StructField('venue',        StringType(), True),
    StructField('innings',      StringType(), True),
    StructField('ball',         StringType(), True),
    StructField('batting_team', StringType(), True),
    StructField('bowling_team', StringType(), True),
    StructField('striker',      StringType(), True),
    StructField('non_striker',  StringType(), True),
    StructField('bowler',       StringType(), True),
    StructField('runs_off_bat', StringType(), True),
    StructField('extras',       StringType(), True),
    StructField('wides',        StringType(), True),
    StructField('noballs',      StringType(), True),
    StructField('byes',         StringType(), True),
    StructField('legbyes',      StringType(), True),
    StructField('penalty',      StringType(), True),
    StructField('wicket_type',  StringType(), True),
    StructField('player_dismissed', StringType(), True),
    StructField('other_wicket_type', StringType(), True),
    StructField('other_player_dismissed', StringType(), True),
])

# all the columns are StringType as Bronze - cast to correct types in Silver
# this ensures no data is ever rejected at ingestion
print(f"Schema defined with {len(delivery_schema.fields)} columns")

# Auto Loader Stream

In [0]:
def load_deliveries_to_bronze() :

    # create database if not exists
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {BRONZE_DB}")

    # create table if not exists
    spark.sql(f"CREATE TABLE IF NOT EXISTS {BRONZE_TABLE}")

    # read stream using Auto Loader
    df_raw = (
        spark.readStream
        .format("cloudFiles") # Auto Loader format
        .option("cloudFiles.format", "csv") # Source file format
        .option("cloudFiles.schemaLocation", SCHEMA_PATH) # Schema tracking
        .option("header", "true") # csvs have header
        .option("rescuedDataColumn", "_rescued_data") # capture unparsed data
        .schema(delivery_schema)
        .load(CRICSHEET_SOURCE_PATH)
    )

    # add metadata columns
    df_enriched = (
        df_raw
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_file_size", col("_metadata.file_size"))
        .withColumn("_file_modified", col("_metadata.file_modification_time"))
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source", lit("cricsheet"))
        # extract match_id from filename: "1234567.csv" -> "1234567"
        .withColumn("_file_match_id", regexp_extract(col("_metadata.file_path"), "(\d+).csv$", 1))
        # fix season format: 2020/21 -> "2020"
        .withColumn("season", col("season").substr(1, 4))
    )

    # write stream to Delta table
    query = (
        df_enriched
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .option("mergeSchema", "true") # handles new columns gracefully
        .trigger(availableNow = True) # process all files then stop- preferred for Batch processing
        .toTable(BRONZE_TABLE)
    )

    query.awaitTermination()
    print("Bronze delivery load complete")

load_deliveries_to_bronze()

# Verify the load

In [0]:
df_bronze = spark.table(BRONZE_TABLE)
print(f"Total rows: {df_bronze.count():,}")

# check metadata columns
df_bronze.select(
    "_source_file",
    "_ingested_at",
    "_source",
    "season",
    "batting_team",
    "runs_off_bat"
).show(5, truncate = False)

# season loaded - should be 2008 to 2024
df_bronze.groupBy("season").count().orderBy("season").show(20)

# check for any rescued (unparseable) rows
rescued = df_bronze.filter(col("_rescued_data").isNotNull())
print(f"Rescued rows (parse failures): {rescued.count()}")

# Optimize the Bronze table

In [0]:
# run after initial load - improves query speed significantly
spark.sql(f"OPTIMIZE {BRONZE_TABLE}")

# check table details
spark.sql(f"DESCRIBE DETAIL {BRONZE_TABLE}").show(vertical = True)

# see table history - this is Delta lake time travel
spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}").show(5)